# Model Evaluation and Final Testing

This is the final stage of the pipeline. Across the previous notebooks we prepared the data, engineered features, and trained **eight models** in total, two algorithms (Random Forest and XGBoost) each trained on four different feature subsets (original, all, uncorrelated, important). Now we evaluate all eight on the held-out **validation set** to compare them, select the best-performing configuration, and finally test it on the completely unseen **test set** to estimate real-world generalisation performance.

## Evaluation Metrics

We use four standard regression metrics and one operational metric:

**MSE** (Mean Squared Error): It's calculated by taking the average of squared prediction errors. It penalises large errors heavily. It's also reported on the **log-price** scale because the target was log-transformed. 
**RMSE** (Root Mean Squared Error): It's the square root of MSE and has same units as the target (log-euros), making it more interpretable than MSE. An RMSE of 0.2 means predictions deviate by roughly 0.2 on the log scale, which corresponds to approximately a 20% error in the original price. 
**MAE** (Mean Absolute Error) It's calculated by taking the average of absolute prediction errors. It's less sensitive to outliers than RMSE. **R2** (R-squared): The proportion of target variance explained by the model. It ranges from 0 to 1. 
**Latency**: Measures the wall-clock time for a single prediction with units in milliseconds.

In [1]:
import joblib
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
from time import time

In [2]:
val_original = pd.read_csv("subsets/val_original.csv")
val_all = pd.read_csv("subsets/val_all.csv")
val_uncorrelated = pd.read_csv("subsets/val_uncorrelated.csv")
val_important = pd.read_csv("subsets/val_important.csv")
val_target = pd.read_csv("subsets/val_target.csv")

## Loading All Trained Models

We load all eight trained models (2 algorithms x 4 feature subsets) from the `models/` directory. They are stored in a dictionary keyed by a combined string like `algorithm_subset` (e.g., `xgb_uncorrelated`), which makes it easy to loop over all combinations without writing repetitive code.


In [3]:
models = {}

for i in ["rf", "xgb"]:
    for j in ["original", "all", "uncorrelated", "important"]:
        models[f"{i}_{j}"] = joblib.load(f"models/model_{i}_{j}.pkl")

## Evaluation Function

We define `evaluate_model()` which takes a model name, the model object, the predictor matrix, and the true target, then computes all four metrics plus inference latency.


In [4]:
results = []

def evaluate_model(name, model, predictors, target):
    start = time()
    pred = model.predict(predictors)
    end = time()
    mse = mean_squared_error(target, pred)
    rmse = root_mean_squared_error(target, pred)
    mae = mean_absolute_error(target, pred)
    r2 = r2_score(target, pred)
    latency = end-start
    results.append({
        'model': name,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'latency': latency*1000
    })
    
    # print(f"{name} -- MSE: {round(mse,3)}, MAE: {round(mae,3)}, R2: {round(r2,3)}, Latency: {latency*1000} ms")

In [5]:
# Evaluate models with original features
evaluate_model("Original_features RF", models["rf_original"], val_original, val_target)
evaluate_model("Original_features XGB", models["xgb_original"], val_original, val_target)

# Evaluate models with all features
evaluate_model("All_features RF", models["rf_all"], val_all, val_target)
evaluate_model("All_features XGB", models["xgb_all"], val_all, val_target)

# Evaluate models with uncorrelated features
evaluate_model("Uncorrelated_features RF", models["rf_uncorrelated"], val_uncorrelated, val_target)
evaluate_model("Uncorrelated_features XGB", models["xgb_uncorrelated"], val_uncorrelated, val_target)

# Evaluate models with important features
evaluate_model("Important_features RF", models["rf_important"], val_important, val_target)
evaluate_model("Important_features XGB", models["xgb_important"], val_important, val_target)

## Validation Results

We collect all results into a single DataFrame for side-by-side comparison.


In [6]:
results = pd.DataFrame(results)
round(results,3)

,model,mse,rmse,mae,r2,latency
0,Original_features RF,0.044,0.209,0.153,0.725,44.228
1,Original_features XGB,0.043,0.208,0.155,0.727,11.998
2,All_features RF,0.041,0.204,0.151,0.738,59.739
3,All_features XGB,0.041,0.203,0.152,0.739,10.002
4,Uncorrelated_features RF,0.044,0.209,0.154,0.723,31.430
5,Uncorrelated_features XGB,0.041,0.203,0.155,0.740,9.198
6,Important_features RF,0.045,0.212,0.157,0.715,53.162
7,Important_features XGB,0.046,0.214,0.159,0.710,5.242


#### Key Observations

**1. XGBoost with uncorrelated features is the clear winner (R2 = 0.74, RMSE = 0.203).** It achieves the lowest MSE, lowest RMSE, lowest MAE, and highest R2 of all eight configurations.

**2. Aggressive feature selection (important features) hurt the most.** Reducing from 26 to just 11 features caused the largest performance drop across both algorithms (R2 fell to ~0.71). While these 11 features are indeed the strongest individual predictors, removing the 'long tail' of weaker features still costs ~2–4 percentage points of R2.

**3. XGBoost is consistently faster than Random Forest.** XGBoost inference takes 5–12 ms compared to RF's 32–55 ms. This is expected: XGBoost typically uses fewer, shallower trees than Random Forest, so each prediction traverses less total tree structure. The speed advantage makes XGBoost the better choice if the model will be deployed in a latency-sensitive application.


---

## Final Test-Set Evaluation

Based on the validation results, we select the **XGBoost model trained on uncorrelated features** as our final model. We now evaluate it on the **test set**. This gives us an unbiased estimate of how the model will perform on future, real-world data.

We define a separate `test_model()` function that prints the results directly, rather than appending to the validation results table. This keeps the two evaluation stages cleanly separated.


In [7]:
def test_model(name, model, predictors, target):
    start = time()
    pred = model.predict(predictors)
    end = time()
    mse = mean_squared_error(target, pred)
    rmse = root_mean_squared_error(target, pred)
    mae = mean_absolute_error(target, pred)
    r2 = r2_score(target, pred)
    latency = end-start
    
    print(f"{name} -- MSE: {round(mse,3)}, RMSE: {round(rmse,3)}, MAE: {round(mae,3)}, R2: {round(r2,3)}, Latency: {latency*1000} ms")

In [8]:
test_uncorrelated = pd.read_csv("subsets/test_uncorrelated.csv")
test_target = pd.read_csv("subsets/test_target.csv")

test_model("Test_features", models["xgb_uncorrelated"], test_uncorrelated, test_target)

Test_features -- MSE: 0.026, RMSE: 0.163, MAE: 0.129, R2: 0.741, Latency: 8.124589920043945 ms


### Interpreting the Final Test Results

**Test set performance:** MSE = 0.026, RMSE = 0.163, MAE = 0.129, R2 = 0.741, Latency = 8.12 ms

**Generalisation check:** The test R2 is very close to the validation R2. In fact, the model performs slightly *better* on the test set (lower MSE, lower RMSE, lower MAE). This is an encouraging sign: it means the model did not overfit to the validation set and generalises well to completely unseen apartments.

**What does R2 = 0.74 mean in practice?** The model explains 74% of the variance in log-rent prices. The remaining 24% is unexplained. This captures factors not in our feature set (exact neighbourhood within a district, building condition, landlord pricing strategy, seasonal demand fluctuations, etc.) as well as inherent noise in rental listings.

**RMSE on the original euro scale:** Since the target was log-transformed, the RMSE of 0.157 on the log scale translates roughly to a percentage error. A prediction with log-error 0.157 corresponds to the model being off by approximately exp(0.157) - 1 = ~17% on average. For example, if the true rent is 500 euros/month, the model's typical prediction would fall between roughly 415 and 585 euros. The MAE of 0.125 is even tighter, suggesting the median error is closer to ~13%.

**Inference speed:** At ~8 ms for a full batch prediction, the model is fast enough for real-time use in a web application or API. Even predicting a single apartment's rent would take well under a millisecond.
